In [1]:
import os
import sys
import re
import glob
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv

In [2]:
load_dotenv()
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TOKENIZERS_PARALLELISM '] = 'false'

In [3]:
from sentence_transformers import SentenceTransformer
import chromadb
import faiss
import pypdf

In [4]:
DATASET_DIR = "clouddesk_dataset"

source_summary = []
for folder in os.listdir(DATASET_DIR):
    folder_path = os.path.join(DATASET_DIR, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        source_summary.append({
            'Folder': folder,
            'File Count': len(files),
            'Sample Files': ', '.join(files[:2]) 
        })

df_audit = pd.DataFrame(source_summary)
try: display(df_audit)
except NameError: print(df_audit)

,Folder,File Count,Sample Files
0,engineering_runbooks,3,"runbook-webhook-delivery-failures.md, runbook-..."
1,release_notes,3,"RN-3.5.0_release-notes.pdf, RN-3.4.0_release-n..."
2,api_documentation,4,"authentication.md, webhooks-reference.md"
3,support_tickets,1,previous_support_tickets.csv
4,help_center_articles,5,"HC-101_integrating-clouddesk-with-slack.pdf, H..."


In [5]:
class Documentchunk:
    def __init__(self, text: str, metadata: dict):
        self.page_content = text.strip()
        self.metadata = metadata
    
    def __repr__(self):
        return f"<Chunk id={self.metadata.get('chunk_id')} type={self.metadata.get('source_type')} title='{self.metadata.get('title')}'>"  

In [6]:
def parse_markdown(filepath: str, source_type: str) -> tuple:
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    meta= {
        'source_type': source_type,
        'title': os.path.splitext(os.path.basename(filepath))[0].replace('-', ' ').title(),
        'last_updated': '2026-09-15',
        'version': 'v3.6.0',
        'source_file': os.path.basename(filepath)
    }
    body = content
    if content.startswith('---'):
        parts = content.split('---', 2)
        if len(parts) >= 3:
            body = parts[2]
            for line in parts[1].strip().split('\n'):
                if ':' in line:
                    k, v = line.split(':', 1)
                    meta[k.strip().lower()] = v.strip().strip('"\'')
    for line in body.split('\n'):
        if line.startswith('# '):
            meta['title'] = line.replace('# ', ' ').strip()
            break      
    return meta, body        

In [7]:
def parse_pdf(filepath: str, source_type: str) -> tuple:
    reader = pypdf.PdfReader(filepath)
    text = ''
    for p in reader.pages:
        text += (p.extract_text() or '') + '\n\n'

    meta= {
        'source_type': source_type,
        'title': os.path.splitext(os.path.basename(filepath))[0].replace('_', ' ').title(),
        'last_updated': '2026-09-15',
        'version': 'v3.6.0',
        'source_file': os.path.basename(filepath)
    }
    for line in text.split('\n')[:10]:
        if 'Version' in line:
            v_m = re.search(r'Version:\s*([w\.]+)', line)
            if v_m: meta['version'] = v_m.group(1)
        if 'Last Updated' in line:
            d_m = re.search(r'Last Updated:\s*([w\.]+)', line)
            if d_m: meta['last_updated'] = d_m.group(1)
    return meta, text
    


In [8]:
chunks = []
counter = 0

for fp in glob.glob(os.path.join(DATASET_DIR, 'api_documentation', '*.md')):
    meta, body = parse_markdown(fp, 'api_documentation')
    for sec in body.split('\n## '):
        counter += 1
        m = meta.copy()
        m['chunk_id'] = f"api_doc_{counter:04d}"
        chunks.append(Documentchunk(sec, m))
    

In [9]:

for fp in glob.glob(os.path.join(DATASET_DIR, 'engineering_runbooks', '*.md')):
    meta, body = parse_markdown(fp, 'engineering_runbooks')
    for sec in body.split('\n## '):
        if len(sec.strip()) >30:
          counter += 1
          m = meta.copy()
          m['chunk_id'] = f"engineering_runbook_{counter:04d}"
          chunks.append(Documentchunk(sec, m))
    

In [10]:

for fp in glob.glob(os.path.join(DATASET_DIR, 'help_center_articles', '*.pdf')):
    meta, text = parse_pdf(fp, 'help_center_articles')
    for p in text.split('\n\n'):
        if len(p.strip()) >40:
           counter += 1
           m = meta.copy()
           m['chunk_id'] = f"help_center_article_{counter:04d}"
           chunks.append(Documentchunk(p, m))
    

In [11]:

for fp in glob.glob(os.path.join(DATASET_DIR, 'release_notes', '*.pdf')):
    meta, text = parse_pdf(fp, 'release_notes')
    for p in text.split('\n\n'):
        if len(p.strip()) >40:
          counter += 1
          m = meta.copy()
          m['chunk_id'] = f"release_note_{counter:04d}"
          chunks.append(Documentchunk(p, m))
    

In [12]:
csv_p = os.path.join(DATASET_DIR, 'support_tickets', 'previous_support_tickets.csv')
df_tickets = pd.read_csv(csv_p)
for _, r in df_tickets.iterrows():
    counter += 1
    tid = str(r['ticket_id'])
    content = f"Ticket [{tid}] Subject:  {r['subject']}\nQuestion: {r['customer_question']}\nResolution: {r['agent_answer']}"
    m = {
        'source_type': 'support_tickets',
        'title': f"Ticket {tid}: {r['subject']}",
        'chunk_id': f"ticket_{tid}",
        'last_updated': str(r['created_at']),
        'version': 'v3.6.0',
        'source_file': 'previous_support_tickets.csv',
        'source_type_reference': str(r.get('source_type_reference', '')),
        'source_title_reference': str(r.get('source_title_reference', ''))
    }
    chunks.append(Documentchunk(content, m))

In [13]:
len(chunks)

97

In [14]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
texts = [c.page_content for c in chunks]
embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)

print('Embedding matrix shape', embeddings.shape)
print(f'vector dimension: {embeddings.shape[1]}')

/Users/onojamiraclegrace/ds_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/onojamiraclegrace/ds_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding matrix shape (97, 384)
vector dimension: 384


In [15]:
VECTOR_DB_DIR = 'vector_db'
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

#1. Index into chromaDB
chroma_client = chromadb.PersistentClient(path=os.path.join(VECTOR_DB_DIR, 'chroma'))

collection = chroma_client.get_or_create_collection(name='clouddesk_support_kb')

ids = [c.metadata['chunk_id'] for c in chunks]
documents = [c.page_content for c in chunks]
metadata = [c.metadata for c in chunks]

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadata,
)

print(f'ChromaDB Indexed: {collection.count()} vectors')

ChromaDB Indexed: 102 vectors


In [16]:

pinecone_key = os.environ.get('PINECONE_API_KEY')
if pinecone_key:
    try:
        from pinecone import Pinecone, ServerlessSpec


        pc = Pinecone(api_key=pinecone_key)

        index_name = os.environ.get('PINECONE_INDEX_NAME', 'clouddesk-support-rag')
        existing = [idx.name for idx in pc.list_indexes()]
        if index_name not in existing:
            pc.create_index(
                name=index_name,
                dimension=384,
                metric='cosine',
                spec=ServerlessSpec(cloud='aws', region='us-east-1')
            )

            pc_index = pc.Index(index_name)
            vectors_to_upsert = []
            for c, emb in zip(chunks, embeddings):
                meta = {k: str(v) for k, v in c.metadata.items()}
                meta['page_content'] = c.page_content[:500]
                vectors_to_upsert.append((c.metadata['chunk_id'], emb.tolist(), meta))

            for i in range(0, len(vectors_to_upsert), 50):
                pc_index.upsert(vectors=vectors_to_upsert[i:i+50])
            print(f'Successfully indexed {len(vectors_to_upsert)} chunks into pinecone')

    except Exception as e:
            print(f'Pinecone indexing notice: {e}')

else:
    print('Pinecone api key not found in environment')


Successfully indexed 97 chunks into pinecone


In [17]:
with open(os.path.join(VECTOR_DB_DIR, 'chunks_store.json'), 'w', encoding='utf-8') as f:
    json.dump([{'page_content': c.page_content, 'metadata': c.metadata} for c in chunks],
     f, indent=2)